# Quickstart: Build and Inspect an `AnnNet`

This notebook introduces the core object, `annnet.AnnNet`.

You will learn how to:

- create a graph
- add vertices, binary edges, and hyperedges
- inspect the incidence matrix and attribute tables
- read the high-level graph summary fields exposed by the API


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PACKAGE_ROOT = (NOTEBOOK_DIR / "..").resolve()
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import annnet as an


In [2]:
G = an.AnnNet(
    directed=None,
    study="toy stress-signaling network",
    organism="human",
)

G.add_vertices(
    [
        {"vertex_id": "ATM", "kind": "kinase"},
        {"vertex_id": "TP53", "kind": "transcription_factor"},
        {"vertex_id": "MDM2", "kind": "ubiquitin_ligase"},
        {"vertex_id": "CDKN1A", "kind": "effector"},
    ]
)

G.add_edge(
    "ATM",
    "TP53",
    edge_id="atm_activates_tp53",
    edge_directed=True,
    weight=1.0,
    relation="activates",
)

G.add_edge(
    "TP53",
    "MDM2",
    edge_id="tp53_induces_mdm2",
    edge_directed=True,
    weight=0.8,
    relation="induces",
)

G.add_hyperedge(
    head=["TP53", "ATM"],
    tail=["CDKN1A"],
    edge_id="stress_program",
    weight=1.2,
    process="stress_response",
)

G


In [3]:
print("shape:", G.shape)
print("vertices:", G.vertices())
print("edges:", G.edges())
print("num vertices:", G.num_vertices)
print("num edges:", G.num_edges)
print("graph metadata:", G.uns)


shape: (4, 3)
vertices: ['ATM', 'TP53', 'MDM2', 'CDKN1A']
edges: ['atm_activates_tp53', 'tp53_induces_mdm2', 'stress_program']
num vertices: 4
num edges: 3
graph metadata: {'study': 'toy stress-signaling network', 'organism': 'human'}


## Incidence matrix and AnnData-style accessors

`AnnNet` stores graph structure as a sparse incidence matrix. The `obs`, `var`, and `uns` properties are the closest analogs to the familiar AnnData interface.


In [4]:
X = G.X()
print(type(X).__name__)
print("matrix shape:", X.shape)
X


dok_matrix
matrix shape: (8, 8)


<Dictionary Of Keys sparse matrix of dtype 'float32'
	with 7 stored elements and shape (8, 8)>

In [5]:
G.obs


vertex_id,kind
str,str
"""ATM""","""kinase"""
"""TP53""","""transcription_factor"""
"""MDM2""","""ubiquitin_ligase"""
"""CDKN1A""","""effector"""


In [6]:
G.var


edge_id,relation,process
str,str,str
"""atm_activates_tp53""","""activates""",null
"""tp53_induces_mdm2""","""induces""",null
"""stress_program""",null,"""stress_response"""


## Edge surface helpers

The graph also exposes convenience helpers that flatten the structure into a more tabular form.


In [7]:
G.edge_list()


[('ATM', 'TP53', 'atm_activates_tp53', 1.0),
 ('TP53', 'MDM2', 'tp53_induces_mdm2', 0.8),
 (None, None, 'stress_program', 1.2)]

In [8]:
G.edges_view()


edge_id,kind,directed,global_weight,source,target,edge_type,head,tail,members,relation,process,effective_weight
str,str,bool,f64,str,str,str,list[str],list[str],list[str],str,str,f64
"""atm_activates_tp53""","""binary""",true,1.0,"""ATM""","""TP53""","""regular""",null,null,null,"""activates""",null,1.0
"""tp53_induces_mdm2""","""binary""",true,0.8,"""TP53""","""MDM2""","""regular""",null,null,null,"""induces""",null,0.8
"""stress_program""","""hyper""",true,1.2,null,null,null,"[""ATM"", ""TP53""]","[""CDKN1A""]",null,null,"""stress_response""",1.2
